# LlamaEdge

[LlamaEdge](https://github.com/second-state/LlamaEdge) 允许您与本地或通过聊天服务以 [GGUF](https://github.com/ggerganov/llama.cpp/blob/master/gguf-py/README.md) 格式的大型语言模型 (LLM) 进行聊天。

- `LlamaEdgeChatService` 为开发者提供了一个与 OpenAI API 兼容的服务，可以通过 HTTP 请求与 LLM 进行聊天。

- `LlamaEdgeChatLocal` 使开发者能够在本地与 LLM 进行聊天（即将推出）。

`LlamaEdgeChatService` 和 `LlamaEdgeChatLocal` 均运行在由 [WasmEdge Runtime](https://wasmedge.org/) 驱动的基础设施上，该运行时为 LLM 推理任务提供了一个轻量级且可移植的 WebAssembly 容器环境。

## 通过 API 服务聊天

`LlamaEdgeChatService` 在 `llama-api-server` 上运行。按照 [llama-api-server quick-start](https://github.com/second-state/llama-utils/tree/main/api-server#readme) 中的步骤操作，您可以托管自己的 API 服务，这样只要有互联网连接，您就可以在任何设备上与任何您喜欢的模型进行聊天。

In [2]:
from langchain_community.chat_models.llama_edge import LlamaEdgeChatService
from langchain_core.messages import HumanMessage, SystemMessage

### 与 LLM 进行非流式聊天

In [5]:
# service url
service_url = "https://b008-54-186-154-209.ngrok-free.app"

# create wasm-chat service instance
chat = LlamaEdgeChatService(service_url=service_url)

# create message sequence
system_message = SystemMessage(content="You are an AI assistant")
user_message = HumanMessage(content="What is the capital of France?")
messages = [system_message, user_message]

# chat with wasm-chat service
response = chat.invoke(messages)

print(f"[Bot] {response.content}")

[Bot] Hello! The capital of France is Paris.


### 使用流式模式与大型语言模型聊天

This guide will show you how to chat with LLMs in streaming mode.
This means that the LLM will respond to your prompts in real-time, as it generates the response.
This is a much more engaging and interactive experience than waiting for the LLM to generate the entire response before displaying it.

本指南将向您展示如何使用流式模式与大型语言模型（LLM）进行聊天。
这意味着LLM将在生成响应的同时，实时地响应您的提示。
与等待LLM生成完整响应后再显示相比，这是一种更具吸引力和交互性的体验。

**Prerequisites**

*   A LangChain installation. If you don't have it installed, you can install it using pip:
    ```bash
    pip install langchain
    ```

**先决条件**

*   已安装LangChain。如果尚未安装，您可以使用pip进行安装：
    ```bash
    pip install langchain
    ```

**Streaming LLM Responses**

There are two main ways to stream LLM responses:

1.  **Using the `stream` method:** This is the most straightforward way to stream LLM responses. The `stream` method returns an iterator that yields chunks of the response as they are generated.

    ```python
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage

    chat = ChatOpenAI(model="gpt-3.5-turbo")

    for chunk in chat.stream([HumanMessage(content="Hello, how are you?")]):
        print(chunk)
    ```

    The output will be a series of `AIMessageChunk` objects, each containing a portion of the response.

    ```
    AIMessageChunk(content=' I am doing well, thank you for asking! How can I help you today?', additional_kwargs={}, example=[])
    ```

2.  **Using the `astream` method:** This method is similar to `stream` but returns an asynchronous iterator. This is useful when you are working with asynchronous code.

    ```python
    import asyncio
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage

    async def main():
        chat = ChatOpenAI(model="gpt-3.5-turbo")
        async for chunk in chat.astream([HumanMessage(content="Hello, how are you?")]):
            print(chunk)

    asyncio.run(main())
    ```

    The output will be the same as the `stream` method, but it will be generated asynchronously.

**流式传输 LLM 响应**

有两种主要方法可以流式传输 LLM 响应：

1.  **使用 `stream` 方法：** 这是流式传输 LLM 响应最直接的方法。`stream` 方法返回一个迭代器，该迭代器在生成响应的各个部分时会产生响应的块。

    ```python
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage

    chat = ChatOpenAI(model="gpt-3.5-turbo")

    for chunk in chat.stream([HumanMessage(content="Hello, how are you?")]):
        print(chunk)
    ```

    输出将是一系列 `AIMessageChunk` 对象，每个对象包含响应的一部分。

    ```
    AIMessageChunk(content=' I am doing well, thank you for asking! How can I help you today?', additional_kwargs={}, example=[])
    ```

2.  **使用 `astream` 方法：** 此方法与 `stream` 类似，但返回一个异步迭代器。这在您使用异步代码时非常有用。

    ```python
    import asyncio
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage

    async def main():
        chat = ChatOpenAI(model="gpt-3.5-turbo")
        async for chunk in chat.astream([HumanMessage(content="Hello, how are you?")]):
            print(chunk)

    asyncio.run(main())
    ```

    输出将与 `stream` 方法相同，但它是异步生成的。

**Customizing the Streaming Experience**

You can also customize the streaming experience by using callbacks. Callbacks allow you to hook into the LLM's generation process and perform custom actions, such as displaying the response in a UI or logging it to a file.

To use callbacks, you need to create a callback handler that inherits from `BaseCallbackHandler`. Then, you can pass an instance of your callback handler to the `stream` or `astream` method.

```python
from typing import Any, Dict, List, Optional, Union
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import AIMessageChunk
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

class MyCustomHandler(BaseCallbackHandler):
    def on_llm_stream(
        self,
        response: List[AIMessageChunk],
        *,
        run_id: str,
        parent_run_id: Optional[str] = None,
        tags: Optional[List[str]] = None,
        metadata: Optional[Dict[str, Any]] = None,
        **kwargs: Any,
    ) -> None:
        """Callback when there's an incoming LLM stream event."""
        for chunk in response:
            print(chunk.content, end="", flush=True)

chat = ChatOpenAI(model="gpt-3.5-turbo")

for chunk in chat.stream([HumanMessage(content="Hello, how are you?")],
                         {"callbacks": [MyCustomHandler()]}):
    pass
```

This will print the response to the console as it is generated, with no extra formatting.

**自定义流式传输体验**

您还可以通过使用回调来自定义流式传输体验。回调允许您挂钩 LLM 的生成过程并执行自定义操作，例如在 UI 中显示响应或将其记录到文件。

要使用回调，您需要创建一个继承自 `BaseCallbackHandler` 的回调处理程序。然后，您可以将回调处理程序的实例传递给 `stream` 或 `astream` 方法。

```python
from typing import Any, Dict, List, Optional, Union
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import AIMessageChunk
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

class MyCustomHandler(BaseCallbackHandler):
    def on_llm_stream(
        self,
        response: List[AIMessageChunk],
        *,
        run_id: str,
        parent_run_id: Optional[str] = None,
        tags: Optional[List[str]] = None,
        metadata: Optional[Dict[str, Any]] = None,
        **kwargs: Any,
    ) -> None:
        """当有传入的 LLM 流事件时回调。"""
        for chunk in response:
            print(chunk.content, end="", flush=True)

chat = ChatOpenAI(model="gpt-3.5-turbo")

for chunk in chat.stream([HumanMessage(content="Hello, how are you?")],
                         {"callbacks": [MyCustomHandler()]}):
    pass
```

这将作为生成的响应打印到控制台，没有额外的格式。

**Streaming with LangChain Expression Language (LCEL)**

LCEL provides a declarative way to build LLM applications. You can easily stream responses from LCEL chains by using the `stream` method.

```python
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "{input}")
])
model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = prompt | model | output_parser

for chunk in chain.stream({"input": "Hello, how are you?"}):
    print(chunk, end="", flush=True)
```

This will stream the response from the chain, including the prompt and the model's output.

**使用 LangChain 表达式语言 (LCEL) 进行流式传输**

LCEL 提供了一种声明式的方法来构建 LLM 应用程序。您可以通过使用 `stream` 方法轻松地从 LCEL 链流式传输响应。

```python
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "{input}")
])
model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = prompt | model | output_parser

for chunk in chain.stream({"input": "Hello, how are you?"}):
    print(chunk, end="", flush=True)
```

这将从链中流式传输响应，包括提示和模型的输出。

**Streaming with RunnableParallel**

You can also stream responses from `RunnableParallel` by using the `stream` method. This will stream the responses from each runnable in parallel.

```python
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt1 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
prompt2 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")

model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = RunnableParallel(
    joke1=prompt1 | model | output_parser,
    joke2=prompt2 | model | output_parser,
)

for chunk in chain.stream({"topic": "cats"}):
    print(chunk)
```

This will stream the responses from both runnables in parallel, and you will see the output as it is generated.

**使用 RunnableParallel 进行流式传输**

您还可以通过使用 `stream` 方法从 `RunnableParallel` 流式传输响应。这将并行流式传输每个可运行对象的响应。

```python
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt1 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
prompt2 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")

model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = RunnableParallel(
    joke1=prompt1 | model | output_parser,
    joke2=prompt2 | model | output_parser,
)

for chunk in chain.stream({"topic": "cats"}):
    print(chunk)
```

这将并行流式传输两个可运行对象的响应，您将看到生成的输出。

**Streaming with RunnableSequence**

You can also stream responses from `RunnableSequence` by using the `stream` method. This will stream the responses from each runnable in the sequence.

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

prompt = ChatPromptTemplate.from_template("Translate the following English text to French: {text}")
model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = RunnableSequence(prompt, model, output_parser)

for chunk in chain.stream({"text": "Hello, world!"}):
    print(chunk, end="", flush=True)
```

This will stream the response from the sequence, and you will see the output as it is generated.

**使用 RunnableSequence 进行流式传输**

您还可以通过使用 `stream` 方法从 `RunnableSequence` 流式传输响应。这将流式传输序列中每个可运行对象的响应。

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

prompt = ChatPromptTemplate.from_template("Translate the following English text to French: {text}")
model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = RunnableSequence(prompt, model, output_parser)

for chunk in chain.stream({"text": "Hello, world!"}):
    print(chunk, end="", flush=True)
```

这将从序列中流式传输响应，您将看到生成的输出。

**Streaming with RunnableLambda**

You can also stream responses from `RunnableLambda` by using the `stream` method. This will stream the responses from the lambda function.

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

def process_text(text):
    return text.upper()

prompt = ChatPromptTemplate.from_template("Translate the following English text to French: {text}")
model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = RunnableLambda(process_text) | prompt | model | output_parser

for chunk in chain.stream({"text": "Hello, world!"}):
    print(chunk, end="", flush=True)
```

This will stream the response from the lambda function, and you will see the output as it is generated.

**使用 RunnableLambda 进行流式传输**

您还可以通过使用 `stream` 方法从 `RunnableLambda` 流式传输响应。这将流式传输 lambda 函数的响应。

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

def process_text(text):
    return text.upper()

prompt = ChatPromptTemplate.from_template("Translate the following English text to French: {text}")
model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = RunnableLambda(process_text) | prompt | model | output_parser

for chunk in chain.stream({"text": "Hello, world!"}):
    print(chunk, end="", flush=True)
```

这将从 lambda 函数流式传输响应，您将看到生成的输出。

**Streaming with RunnablePassthrough**

You can also stream responses from `RunnablePassthrough` by using the `stream` method. This will stream the responses from the passthrough.

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("Translate the following English text to French: {text}")
model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = RunnablePassthrough() | prompt | model | output_parser

for chunk in chain.stream({"text": "Hello, world!"}):
    print(chunk, end="", flush=True)
```

This will stream the response from the passthrough, and you will see the output as it is generated.

**使用 RunnablePassthrough 进行流式传输**

您还可以通过使用 `stream` 方法从 `RunnablePassthrough` 流式传输响应。这将流式传输 passthrough 的响应。

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("Translate the following English text to French: {text}")
model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = RunnablePassthrough() | prompt | model | output_parser

for chunk in chain.stream({"text": "Hello, world!"}):
    print(chunk, end="", flush=True)
```

这将从 passthrough 流式传输响应，您将看到生成的输出。

**Streaming with RunnableMap**

You can also stream responses from `RunnableMap` by using the `stream` method. This will stream the responses from each runnable in the map.

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

prompt1 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
prompt2 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")

model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = RunnableMap({
    "joke1": prompt1 | model | output_parser,
    "joke2": prompt2 | model | output_parser,
})

for chunk in chain.stream({"topic": "cats"}):
    print(chunk)
```

This will stream the responses from both runnables in the map, and you will see the output as it is generated.

**使用 RunnableMap 进行流式传输**

您还可以通过使用 `stream` 方法从 `RunnableMap` 流式传输响应。这将流式传输 map 中每个可运行对象的响应。

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

prompt1 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
prompt2 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")

model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = RunnableMap({
    "joke1": prompt1 | model | output_parser,
    "joke2": prompt2 | model | output_parser,
})

for chunk in chain.stream({"topic": "cats"}):
    print(chunk)
```

这将流式传输 map 中两个可运行对象的响应，您将看到生成的输出。

**Streaming with RunnableBranch**

You can also stream responses from `RunnableBranch` by using the `stream` method. This will stream the responses from the branch.

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch

prompt1 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
prompt2 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")

model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = RunnableBranch(
    (lambda x: x["topic"] == "cats", prompt1 | model | output_parser),
    (lambda x: x["topic"] == "dogs", prompt2 | model | output_parser),
    (lambda x: True, prompt1 | model | output_parser), # Default case
)

for chunk in chain.stream({"topic": "cats"}):
    print(chunk)
```

This will stream the response from the branch, and you will see the output as it is generated.

**使用 RunnableBranch 进行流式传输**

您还可以通过使用 `stream` 方法从 `RunnableBranch` 流式传输响应。这将流式传输分支的响应。

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch

prompt1 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
prompt2 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")

model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = RunnableBranch(
    (lambda x: x["topic"] == "cats", prompt1 | model | output_parser),
    (lambda x: x["topic"] == "dogs", prompt2 | model | output_parser),
    (lambda x: True, prompt1 | model | output_parser), # Default case
)

for chunk in chain.stream({"topic": "cats"}):
    print(chunk)
```

这将从分支流式传输响应，您将看到生成的输出。

**Streaming with RunnableWithFallbacks**

You can also stream responses from `RunnableWithFallbacks` by using the `stream` method. This will stream the responses from the fallbacks.

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableWithFallbacks

prompt = ChatPromptTemplate.from_template("Translate the following English text to French: {text}")
model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

fallback_chain = ChatPromptTemplate.from_template("Sorry, I can't translate that. Please try again.") | model | output_parser

chain = RunnableWithFallbacks(
    fallback=fallback_chain,
    runnable=prompt | model | output_parser
)

for chunk in chain.stream({"text": "Hello, world!"}):
    print(chunk, end="", flush=True)
```

This will stream the response from the fallbacks, and you will see the output as it is generated.

**使用 RunnableWithFallbacks 进行流式传输**

您还可以通过使用 `stream` 方法从 `RunnableWithFallbacks` 流式传输响应。这将流式传输备用项的响应。

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableWithFallbacks

prompt = ChatPromptTemplate.from_template("Translate the following English text to French: {text}")
model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

fallback_chain = ChatPromptTemplate.from_template("Sorry, I can't translate that. Please try again.") | model | output_parser

chain = RunnableWithFallbacks(
    fallback=fallback_chain,
    runnable=prompt | model | output_parser
)

for chunk in chain.stream({"text": "Hello, world!"}):
    print(chunk, end="", flush=True)
```

这将从备用项流式传输响应，您将看到生成的输出。

**Streaming with RunnableConfig**

You can also stream responses from `RunnableConfig` by using the `stream` method. This will stream the responses from the config.

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableConfig

prompt = ChatPromptTemplate.from_template("Translate the following English text to French: {text}")
model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = prompt | model | output_parser

for chunk in chain.stream({"text": "Hello, world!"}, config=RunnableConfig(run_name="my_run")):
    print(chunk, end="", flush=True)
```

This will stream the response from the config, and you will see the output as it is generated.

**使用 RunnableConfig 进行流式传输**

您还可以通过使用 `stream` 方法从 `RunnableConfig` 流式传输响应。这将流式传输配置的响应。

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableConfig

prompt = ChatPromptTemplate.from_template("Translate the following English text to French: {text}")
model = ChatOpenAI(model="gpt-3.5-turbo")
output_parser = StrOutputParser()

chain = prompt | model | output_parser

for chunk in chain.stream({"text": "Hello, world!"}, config=RunnableConfig(run_name="my_run")):
    print(chunk, end="", flush=True)
```

这将从配置流式传输响应，您将看到生成的输出。

**Streaming with RunnableAsyncConfig**

You can also stream responses from `RunnableAsyncConfig` by using the `stream` method. This will stream the responses from the async config.

```python
import asyncio
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableAsyncConfig

async def main():
    prompt = ChatPromptTemplate.from_template("Translate the following English text to French: {text}")
    model = ChatOpenAI(model="gpt-3.5-turbo")
    output_parser = StrOutputParser()

    chain = prompt | model | output_parser

    async for chunk in chain.astream({"text": "Hello, world!"}, config=RunnableAsyncConfig(run_name="my_async_run")):
        print(chunk, end="", flush=True)

asyncio.run(main())
```

This will stream the response from the async config, and you will see the output as it is generated.

**使用 RunnableAsyncConfig 进行流式传输**

您还可以通过使用 `stream` 方法从 `RunnableAsyncConfig` 流式传输响应。这将流式传输异步配置的响应。

```python
import asyncio
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableAsyncConfig

async def main():
    prompt = ChatPromptTemplate.from_template("Translate the following English text to French: {text}")
    model = ChatOpenAI(model="gpt-3.5-turbo")
    output_parser = StrOutputParser()

    chain = prompt | model | output_parser

    async for chunk in chain.astream({"text": "Hello, world!"}, config=RunnableAsyncConfig(run_name="my_async_run")):
        print(chunk, end="", flush=True)

asyncio.run(main())
```

这将从异步配置流式传输响应，您将看到生成的输出。

In [4]:
# service url
service_url = "https://b008-54-186-154-209.ngrok-free.app"

# create wasm-chat service instance
chat = LlamaEdgeChatService(service_url=service_url, streaming=True)

# create message sequence
system_message = SystemMessage(content="You are an AI assistant")
user_message = HumanMessage(content="What is the capital of Norway?")
messages = [
    system_message,
    user_message,
]

output = ""
for chunk in chat.stream(messages):
    # print(chunk.content, end="", flush=True)
    output += chunk.content

print(f"[Bot] {output}")

[Bot]   Hello! I'm happy to help you with your question. The capital of Norway is Oslo.
